In [10]:
import pandas as pd
import os
import probabilistic_automaton as pa # to create the probabilistic automaton for sampling
import vectorization as vec # to create entropy vectors for data encoding/preprocessing
import sampling 

import itertools
from pyranges.readers import read_gtf
from processing_fasta import *
from collections import defaultdict

from Bio import SeqIO

from torch.utils.data import DataLoader
import training as tr
import torch.nn as nn
import torch.optim as optim
import torch






In [5]:
import sys
print(sys.executable)

/root/VRAIN_project/.venv/bin/python


In [2]:

gtf_file_path = '/data/gencode.v49lift37.basic.annotation_protein_coding.gtf' # to build on the automata
cwd = Path(os.getcwd())
print("Loading GTF database into memory, please wait...")
gtf = read_gtf(str(cwd) + gtf_file_path, as_df=True)
print("GTF loaded successfully!")

Loading GTF database into memory, please wait...
GTF loaded successfully!


In [3]:

#Getting the number of genes the 22 chromosomes and the X and Y chromosomes
chr_list = ['chr1', 'chr2', 'chr3', 'chr4', 'chr5', 'chr6', 'chr7', 'chr8', 'chr9',
            'chr10', 'chr11', 'chr12', 'chr13', 'chr14', 'chr15', 'chr16', 'chr17', 'chr18', 'chr19', 'chr20', 'chr21', 'chr22', 'chrX', 'chrY']
for chr in chr_list:
    gtf_filtered = gtf[gtf['Chromosome'] == chr]
    number_genes = len(gtf_filtered)
    print(f'number of genes on {chr}: ', number_genes)
    

number of genes on chr1:  2098
number of genes on chr2:  1257
number of genes on chr3:  1085
number of genes on chr4:  762
number of genes on chr5:  897
number of genes on chr6:  1049
number of genes on chr7:  937
number of genes on chr8:  710
number of genes on chr9:  797
number of genes on chr10:  743
number of genes on chr11:  1318
number of genes on chr12:  1042
number of genes on chr13:  321
number of genes on chr14:  619
number of genes on chr15:  607
number of genes on chr16:  860
number of genes on chr17:  1190
number of genes on chr18:  267
number of genes on chr19:  1483
number of genes on chr20:  549
number of genes on chr21:  230
number of genes on chr22:  449
number of genes on chrX:  876
number of genes on chrY:  63


## General Sampling: first try ( code too heavy not to run)
This code takes in: list of chromosome names 
what it does : Creates a folder named "generated_sequences" 
               Creates a folder for each Chromosome in "generated_sequences" 
               For each Chromosome : gets the number of its genes 
                                    For each gene: creates an automata for that gene 
                                                   generates N samples for that gene 
                                                   saves each sample in fasta file in the chromosome's folder
Return: CSV file with the following structure:sample_id, chromosome, gene_location, sequence_length, acceptance_status, acceptance_score, file_path

Test1: Running the code only on Chromosome 22
       The chromosome has: 449 genes
       Trying to Generate N = 100 samples for each gene
       ((( 449 automata and 44 900 generation )))
Results: more than 30 minutes of running 

Test2: Running the code only on Chromosome 22
       The chromosome has: 449 genes BUT WE SELECT THE FIRST 10 GENES 
       Trying to Generate N = 100 samples for each gene
Results: 3 minutes of running. Samples well Generated

Solution? Try MultiProcessing not sure there are no matrix

In [ ]:
database_rows = [] #CSV file 
chr_list = ['chr22'] # for the moment running only on chr22, but we can run it on all chromosomes later

for chr in chr_list:
    os.makedirs(f"generated_sequences/{chr}", exist_ok=True) #folder for each chromosome
    
    #Getting the number of genes on this chromosome
    gtf_filtered = gtf[gtf['Chromosome'] == chr]
    number_genes = len(gtf_filtered)
    
    #Generating 100 samples for each gene on the chromosome
    for location in range(1, number_genes + 1): 
        my_automata = pa.automata_builder(gtf_file_path, chr, gene_number = location) # creating the automata
        N = 5 # number of samples to generate per gene
        for i in range(N):
            sample = sampling.mutated_sample(id = i, chromosome = chr, location = location, sequence = [], automata = my_automata)
            sample.generate_mutated_sample()
            acceptance_status, score = my_automata.accepts(sample.sequence)

            # create a text file for the generated sequence
            file_name = f"{location}_sample_{i}.fasta"
            file_path = f"generated_sequences/{chr}/{file_name}"
            # save the generated sequence in a fasta file
            with open(file_path, "w") as f:
                f.write(f">{location}_sample_{i}\n") # Standard FASTA header
                f.write(sample.sequence)
            
            # save the metadata AND the file path to our database list
            database_rows.append({
                "sample_id": i,
                "chromosome": chr,
                "gene_location": location, 
                "sequence_length": len(sample.sequence),
                "acceptance_status": acceptance_status,
                "acceptance_score": score,
                "file_path": f"{chr}/{file_name}"
            })

            # export the clean, readable database to a CSV so we can use it after 
            df = pd.DataFrame(database_rows)
            df.to_csv("mutated_samples_database.csv", index=False)
            
print(f"The dataset for th {chr_list} is ready!")



## Focused Sampling : Chromosome 22 gene number 10 ==> 5 000 samples 
## Approach 1:  Without controling the length of the samples 
Why focused sampling ? We are choosing one single gene from one Chromosome 
                       Generating N = 5 000 sample for that gene 

Reason : To run the CNN we need more than N = 100 sample for one single gene.
         

In [4]:

database_rows = [] #CSV file 
chr_list = ['chr22'] # for the moment running only on chr22, but we can run it on all chromosomes later
N = 5000 # number of samples to generate per gene

for chr in chr_list:
    os.makedirs(f"focused_sampling/{chr}", exist_ok=True) #folder for each chromosome
    
    #Getting the number of genes on this chromosome
    gtf_filtered = gtf[gtf['Chromosome'] == chr]
    number_genes = len(gtf_filtered)
    
    # Choosing a specific gene to focus on, for example gene number 10
    focused_gene_number = 10
    location = focused_gene_number
 
    # the one and only automata
    my_automata = pa.automata_builder(gtf_file_path, chr, gene_number = location) # creating the automata
    
    for i in range(N):
        sample = sampling.mutated_sample(id = i, chromosome = chr, location = location, sequence = [], automata = my_automata,status = True)
        # No length Control
        sample.generate_mutated_sample()
        acceptance_status, score = my_automata.accepts(sample.sequence)

        # create a text file for the generated sequence
        file_name = f"{location}_sample_{i}.fasta"
        file_path = f"focused_sampling/{chr}/{file_name}"
        # save the generated sequence in a fasta file
        with open(file_path, "w") as f:
                f.write(f">{location}_sample_{i}\n") # Standard FASTA header
                f.write(sample.sequence)
                
        # save the metadata AND the file path to our database list
        database_rows.append({
                        "sample_id": i,
                        "chromosome": chr,
                        "gene_location": location, 
                        "sequence_length": len(sample.sequence),
                        "sample_status": sample.status, # if using controlled length used the status can be false 
                        "acceptance_score": score,
                        "file_path": f"{chr}/{file_name}"
                })

    # export the clean, readable database to a CSV so we can use it after 
    df = pd.DataFrame(database_rows)
    ## named manually 
    df.to_csv("focused_sampling_chr22_g10.csv", index=False)
            
print(f"The dataset for the chromosome(s) {chr_list} gene number {location} is ready! with {N} samples")


KeyboardInterrupt: 

In [5]:
test3 = pd.read_csv("mutated_samples_database.csv")  
print(test3.head())

   sample_id chromosome  gene_location  sequence_length  acceptance_status  \
0          0      chr22             10             5999               True   
1          1      chr22             10            16272               True   
2          2      chr22             10            10718               True   
3          3      chr22             10            21867               True   
4          4      chr22             10              499               True   

   acceptance_score                file_path  
0       -906.322373  chr22/10_sample_0.fasta  
1      -2302.315596  chr22/10_sample_1.fasta  
2      -1552.326170  chr22/10_sample_2.fasta  
3      -3197.410436  chr22/10_sample_3.fasta  
4        -70.761060  chr22/10_sample_4.fasta  


In [6]:
# Check different lengths of the generated samples
test3["sequence_length"].value_counts()

sequence_length
34       67
947       2
15340     2
8720      2
2269      2
         ..
3282      1
19145     1
37465     1
26886     1
4979      1
Name: count, Length: 924, dtype: int64

In [12]:
# Filter on the length problem Check
counts = test3['sequence_length'].value_counts()

rep_len = counts[counts >= 2].index #extracting the lengths that are repititive (from all lengths)

filtered_df = test3[test3['sequence_length'].isin(rep_len)] 
print("the list of repetitive lengths is:")
print (rep_len)

the list of repetitive lengths is:
Index([34, 947, 15340, 8720, 2269, 8093, 16864, 50939, 18880, 30194, 1875], dtype='int64', name='sequence_length')


## Samples Repitition Diagnostic
- Samples of the same length can be doublons
- Run the check all the samples of the same length  
- Eliminate the doublons of samples of the same length
- In order to read fasta files we use the Biopython Library with implemented fonctionalities SeqIo.
#### Method: 
we create a dictionary with this format { "sequences" : ["file1.fasta", "file2.fasta"] }
meaning that the index dequence is represnted by multiple files 
#### Application:
on each repetitive length that we detect in the filtered_reptitive_length dataset.

In [ ]:
df1 = test3 # copy of the dataframe 
print("\n-----------------------------------------------------  Samples Repitition Diagnostic ----------------------------------------------------------")
for length in rep_len:
    filtered = test3.query(f"sequence_length == {length}") # dataframe with only sequences with the same length

    # dictionary regrouping files that present the same sequence
    files_by_sequence = {} 

    for file_name in filtered["file_path"]:
        
        # we join the folder name with the file_name : "./focused_sampling/nom_fichier.fasta"
        complete_path = os.path.join("./focused_sampling", file_name)
        
        try:
            
            # putting the sequence in a tuple and assigning it as a key to the dictionary 
            file_seq_i = tuple(str(record.seq).upper() for record in SeqIO.parse(complete_path, "fasta"))
            
            if file_seq_i in files_by_sequence:
                # if we alr saw this sequence we add its file_name to the list
                files_by_sequence[file_seq_i].append(file_name)
            else:
                # its the first time that we see this file name
                files_by_sequence[file_seq_i]= [file_name]
                
        except FileNotFoundError:
            print(f"file not found : {complete_path}")
            

        
#---------------------------Results---------------------------------------------------------------------------------------------------------------------------

    print(f" The length of {length} we have {len(files_by_sequence)} unique sequences vs {counts[length]} generated sequences  ")
    uniques_only = True

    for sequence, list in files_by_sequence.items():
        if len(list) > 1:
            tous_uniques = False
            print(f" These {len(list)} files are exactly the same")
            print(f"   -> {', '.join(list)}")
#-------------------------- Elimination of Detected Doublons from the datframe -----------------------------------------------------------------------------
            for i in range(len(list)):
                 complete_path = os.path.join("./focused_sampling", file_name)
                if os.path.exists(list[i]):
                     os.remove(list[i])
                     
                df = df[df["file_path"] != list[i]] # we remove the row from the df 
            

    if tous_uniques:
        print("The files are all unique")  
# having new inexed dataframe
df1 = df1.reset_index(drop=True)  


-----------------------------------------------------  Samples Repitition Diagnostic ----------------------------------------------------------
 The length of 34 we have 64 unique sequences vs 67 generated sequences  
 These 4 files are exactly the same
   -> chr22/10_sample_134.fasta, chr22/10_sample_451.fasta, chr22/10_sample_582.fasta, chr22/10_sample_832.fasta
 The length of 947 we have 2 unique sequences vs 2 generated sequences  
 The length of 15340 we have 2 unique sequences vs 2 generated sequences  
 The length of 8720 we have 2 unique sequences vs 2 generated sequences  
 The length of 2269 we have 2 unique sequences vs 2 generated sequences  
 The length of 8093 we have 1 unique sequences vs 2 generated sequences  
 These 2 files are exactly the same
   -> chr22/10_sample_147.fasta, chr22/10_sample_728.fasta
 The length of 16864 we have 2 unique sequences vs 2 generated sequences  
 The length of 50939 we have 2 unique sequences vs 2 generated sequences  
 The length of 18

##  Approach 2: Length Controlled Sampling 5 000 samples 1 000 long. 
###  All the samples are 1 000 Nucleotides long
Methodology :
samples that failed to attend 1000 on a final state have status == false 
to be removed from the dataset 
- results all the samples with 1k of length are status true 
- the other are status false 

We have a Pandas Dataframe holding : ..............
And a list of generated instances of the mutated_sample class ( to use for the next step)

In [3]:

database_fields = [] #CSV file 
chr_list = ['chr22'] # for the moment running only on chr22, but we can run it on all chromosomes later
N = 5000 # number of samples to generate per gene
target_length = 1000

for chr in chr_list:
    os.makedirs(f"focused_sampling_length_control/{chr}", exist_ok=True) #folder for each chromosome
    
    #Getting the number of genes on this chromosome
    gtf_filtered = gtf[gtf['Chromosome'] == chr]
    number_genes = len(gtf_filtered)
    
    # Choosing a specific gene to focus on, for example gene number 10
    focused_gene_number = 10
    location = focused_gene_number
    
    
    my_automata = pa.automata_builder(gtf_file_path, chr, gene_number = location) # creating the automata
    
    list_mutated_samples = [] # list to store the mutated samples ( to store the instances of the mutated_sample class)
    
    for i in range(N):
        sample = sampling.mutated_sample(id = i, chromosome = chr, location = location, sequence = [], automata = my_automata, status = True)
        
        # ------------------------------ With Length control ------------------------------------------
        sample.generate_mutated_sample_length_control(target_length) #status updated here
        
        acceptance_status, score = my_automata.accepts(sample.sequence)

        # create a text file for the generated sequence
        file_name = f"{location}_sample_{i}.fasta"
        file_path = f"focused_sampling_length_control/{chr}/{file_name}"
        # save the generated sequence in a fasta file
        with open(file_path, "w") as f:
                f.write(f">{location}_sample_{i}\n") # Standard FASTA header
                f.write(sample.sequence)
                
        # save the metadata AND the file path to our database list
        database_fields.append({
                        "sample_id": i,
                        "chromosome": chr,
                        "gene_location": location, 
                        "sequence_length": len(sample.sequence),
                        "sample_status": sample.status, # if using controlled length used the status can be false 
                        "acceptance_score": score,
                        "file_path": f"{chr}/{file_name}"
                })
        list_mutated_samples.append(sample) # we store the instance of the mutated_sample class in the list
        
    # export the clean, readable database to a CSV so we can use it after 
    df_length_control = pd.DataFrame(database_fields)
    ## named manually 
    df_length_control.to_csv("focused_sampling_chr22_g10.csv", index=False)
            
print(f"The dataset for the chromosome(s) {chr_list} gene number {location} is ready! \n with {N} samples and target length: {target_length}")


The dataset for the chromosome(s) ['chr22'] gene number 10 is ready! 
 with 5000 samples and target length: 1000


It is faster than the free generation approach.

In [4]:
print(df_length_control.head())
true_only = df_length_control[df_length_control["sample_status"]== True]#those that have status = true
counts = true_only['sequence_length'].value_counts()

# new list of mutated_samples instances  with only the valid samples ( status = True)
list_mutated_samples_valid = [sample for sample in list_mutated_samples if sample.status == True]

print("-----------------------------------------------------Value counts on the True Status samples -----------------------------------------------------")
print(counts)
number_valid_samples = counts[1000]
print("---------------------------------------------------------Sampling Length Control Results -------------------------------------------------------------------")
print(f"                                          We generated {number_valid_samples} 1k-length valid samples from the whole 5k aim                                                 ")

   sample_id chromosome  gene_location  sequence_length  sample_status  \
0          0      chr22             10             1000           True   
1          1      chr22             10             1000           True   
2          2      chr22             10             1000           True   
3          3      chr22             10             1000           True   
4          4      chr22             10             1000           True   

   acceptance_score                file_path  
0       -140.075778  chr22/10_sample_0.fasta  
1       -143.489398  chr22/10_sample_1.fasta  
2       -124.473191  chr22/10_sample_2.fasta  
3       -147.478382  chr22/10_sample_3.fasta  
4       -125.755253  chr22/10_sample_4.fasta  
-----------------------------------------------------Value counts on the True Status samples -----------------------------------------------------
sequence_length
1000    4554
Name: count, dtype: int64
---------------------------------------------------------Sampling Lengt

In [5]:
#Lets check the false status samples how they be acting like trump 
false_only = df_length_control[df_length_control["sample_status"]== False]#those that have status = true
counts = false_only['sequence_length'].value_counts()
print(false_only.head())
print(counts)

    sample_id chromosome  gene_location  sequence_length  sample_status  \
18         18      chr22             10              182          False   
22         22      chr22             10              513          False   
24         24      chr22             10               34          False   
29         29      chr22             10               34          False   
36         36      chr22             10               34          False   

    acceptance_score                 file_path  
18        -35.802468  chr22/10_sample_18.fasta  
22        -77.574749  chr22/10_sample_22.fasta  
24         -1.772589  chr22/10_sample_24.fasta  
29         -1.772589  chr22/10_sample_29.fasta  
36         -1.772589  chr22/10_sample_36.fasta  
sequence_length
34     288
881      3
963      3
600      2
108      2
      ... 
327      1
902      1
104      1
287      1
246      1
Name: count, Length: 147, dtype: int64


## **From a DNA Sequence to an Entropy Vector:**  
### **(Data Encoding - Data Preparation for Neural Networks Training)**
Using the vectorization.py file where we introduced a new class: 
the entropy_vector holding a sample: sampling.mutated_sample, entropies: the vector
Using the entropy_measures.py file:
Calling the `calculate_entropy_float(kmers:list)` function

#### **Vectorization Methodology**

When encoding a DNA sequence $S = \{s_1, s_2, \dots, s_n\}$ of length $N = 1000$ (indices 0 to 999) into an **Entropy Vector**, we map the sequence into a lower-dimensional numerical representation suitable for neural network training:

1. **Window Segmentation:** The 1000-base-pair sequence is partitioned into **10 non-overlapping windows**, each of length 100 (`window_length = 100`).

2. **$k$-mer Frequency Distribution:** Within each 100-bp window, the sequence is analyzed by extracting contiguous substrings of length $k$ (e.g., $k=3$ or $k=10$).

3. **Local Shannon Entropy ($\mathcal{H}$):** For each window $i$, we compute the Shannon entropy of its observed $k$-mer distribution:
   $$\mathcal{H}_i = -\sum_{j} p_j \log_2(p_j)$$
   where $p_j$ is the observed frequency of the $j$-th $k$-mer in window $i$.

---

#### **What Each Vector Field ($v_i$) Represents**
The output is a **10-dimensional feature vector** $[v_1, v_2, \dots, v_{10}]$ where each field represents the sequence complexity ($\mathcal{H}$) of a specific genomic region:

| Vector Field | Genomic Window (Base Pairs) | Description |
| :--- | :--- | :--- |
| **$v_1$** | `[0 : 100]` | Local $k$-mer entropy $\mathcal{H}_1$ of window 1 |
| **$v_2$** | `[100 : 200]` | Local $k$-mer entropy $\mathcal{H}_2$ of window 2 |
| **$v_3$** | `[200 : 300]` | Local $k$-mer entropy $\mathcal{H}_3$ of window 3 |
| **...** | ... | ... |
| **$v_{10}$** | `[900 : 1000]` | Local $k$-mer entropy $\mathcal{H}_{10}$ of window 10 |

> **Interpretation:** > * **High entropy ($v_i \to \text{max}$):** Indicates high sequence complexity/randomness (diverse $k$-mers).  
> * **Low entropy ($v_i \to 0$):** Indicates repetitive or highly conserved DNA motifs within that window.

In [6]:
print("-----------------------------------------Encoding: DNA Sequences ==> Entropy Vectors -----------------------------------------------------")
print(f"We have {len(true_only)} valid samples to encode.")
print(" As a result we will have a list of entropy_vector objects ")
print(f"for example the sequence of the first valid sample instance \n {list_mutated_samples_valid[0].sequence} ")


window_length = 100
#=== TEST 1 : k = 10 ===
k1 = 10 

test1_entropy_vectors = [] # list to store the entropy vectors for all the dataset 

# creating a list of entropy vectors from the list of valid mutated_sample class objects
for sample in list_mutated_samples_valid:
    # creating an entropy_vector object: holding entropies and the sample instance
    sample_vector = vec.entropy_vector(sample, [])
    # this method is called from vectorization.py 
    sample_vector.vectorize( window_length, k1) 
    test1_entropy_vectors.append(sample_vector) # List of all the vectors instances of the dataset

#=== TEST 2 : k = 3 === 
k2 = 3
test2_entropy_vectors = [] 
for sample in list_mutated_samples_valid:
    # creating an entropy_vector object: holding entropies and the sample instance
    sample_vector = vec.entropy_vector(sample, [])
    # this method is called from vectorization.py 
    sample_vector.vectorize( window_length, k2) 
    test2_entropy_vectors.append(sample_vector)
    
    
    

print(f"============== TEST 1: k = 10 ================================")
print(f"example of the first entropy vector for the first valid sample instance \n {test1_entropy_vectors[0].entropies} ")


print(f"============== TEST 2: k = 3 ================================")
print(f"example of the first entropy vector for the first valid sample instance \n {test2_entropy_vectors[0].entropies} ")

-----------------------------------------Encoding: DNA Sequences ==> Entropy Vectors -----------------------------------------------------
We have 4554 valid samples to encode.
 As a result we will have a list of entropy_vector objects 
for example the sequence of the first valid sample instance 
 AAGCACTGGCATTTTTACTTTCATAAACCCTGCCTGAAATAGACATTCATTTGACTTTTCAATTATGGTTTAAAACTAAGTATAAAATATGCTAGAATAAAGCTTCTTCAACAGGTGCAGGCACTCAACAATCATACTGAAACTAATTTTTGCATTTCTACTAGCAATCTATGAGAACTCTCTATATTATCATAATAAAACTGTCAAACTGTTTTCCAACATGATTGTGAGGCCTCCCCTGACATTGTTTATGCTTGCTTAGATACCTGGAGCTCAGAGGGCTGTTGAGGGAGGTAGGCTGGCTTCTGCCAGAGTGACCAGGGCAACTGGACAACTGTAATGTGACCCTCTATCCATATTCTATCCATTATGTTAAGAGTATTGGTAGTTTATTTAATTATTAGTTATGAGAGTTCTTTATATACTCTCCATACAAGTCTTTAATCAACTAAAATCATGCCTTCTCGACAGTGCCCCAAAGTCTTAAAAATCAAGGTGTCATCTGGTGATATGAGTTGGATATGAGATAAAGCTTCTTCAACTATCTGCCCCGTACCTCTAGGACTCCAATTTCACTTTTAAGCAATTAACAGTTTATATTGTTTTGATATTAAAAGGTTTTTAAACTTGGTCTTTCACAGATTATATTGAGTAAATGAACAGCCGTTATTATCCTGAGGGAAATGTTGTTATTGGAAACT

## **Data Labelization based on: Acceptance Score Threshold** 

To train our Convolutional Neural Network (CNN) on the extracted local entropy vectors, we must establish a mathematically rigorous labeling scheme that aligns with formal grammatical inference.

By applying an empirical threshold $\theta$ to this logarithmic score, we automatically partition our sequence database into two distinct biological classes:
*   **Label `1` (Typical / Positive Class)**
*   **Label `0` (Atypical / Counter-Examples)** 

### Formatting the 1-D Grid Topology for PyTorch
To satisfy the multi-channel requirements of PyTorch's 1-D convolutional layer (`nn.Conv1d`), we shape our raw 10-dimensional entropy profiles using `.unsqueeze(0)` to obtain a shape of `(1, 10)`:
$$\text{Tensor Layout: } (\text{in\_channels} = 1, \text{sequence\_length} = 10)$$



In [7]:
# ======= TEST Verify Dataset Shapes and Log-Likelihood Labelization

# list of active instances is named 'list_entropy_vectors'
list_entropy_vectors = test2_entropy_vectors # we can choose either test1 or test2 entropy vectors for the dataset
# We will use an empirical threshold of -150.0 for this test
try:
    test_dataset = tr.DNAEntropyDataset(list_entropy_vectors, threshold=-150.0) #training.py
    
    # We load a small batch size of 4 to inspect the shapes easily
    test_loader = DataLoader(test_dataset, batch_size=4, shuffle=True)

    #  Extract a single mini-batch
    for batch_x, batch_y in test_loader:
        print("Data batch loaded successfully!")
        print("-" * 50)
        print(f"Input batch shape (x)  : {batch_x.shape}")
        print(f"Target batch shape (y) : {batch_y.shape}")
        print("-" * 50)
        
        # verify values
        print(f"Sample continuous entropy vector (shape: {batch_x.shape}):")
        print(batch_x)
        print(f"Sample target classification label : {batch_y.tolist()} (1=Mutated, 0=non-mutated)")
        print("-" * 50)
        
        # Exit after the first batch inspection
        break

except NameError as e:
    print("Ensure 'DNAEntropyDataset' and your list of instances (e.g., 'list_entropy_vectors') are defined above.")

Data batch loaded successfully!
--------------------------------------------------
Input batch shape (x)  : torch.Size([4, 1, 10])
Target batch shape (y) : torch.Size([4])
--------------------------------------------------
Sample continuous entropy vector (shape: torch.Size([4, 1, 10])):
tensor([[[5.4025, 5.3667, 5.2993, 5.4259, 4.5465, 5.2963, 5.1770, 5.2199,
          5.3513, 5.4942]],

        [[5.4473, 4.6470, 5.0434, 5.4125, 5.2687, 5.2870, 5.0519, 5.4155,
          5.5571, 5.1492]],

        [[5.3182, 5.3045, 5.1840, 5.3386, 5.2218, 5.0562, 5.0721, 5.0888,
          5.2064, 5.2987]],

        [[5.3948, 5.3162, 4.9039, 5.3784, 4.9790, 5.0504, 5.3660, 5.2703,
          5.2868, 5.0634]]])
Sample target classification label : [1, 0, 1, 1] (1=Mutated, 0=non-mutated)
--------------------------------------------------


## **Training a Simple 1D CNN Architecture**

We begin with a minimalist baseline model (`DNAEntropyCNN_Simple` imported from `training.py`) to establish an initial benchmark for sequence classification.

### 1/ Experimental Setup & Data Splitting
* **Architecture:** 1D Convolutional Neural Network with a single convolutional layer (`Conv1d`), an activation layer (`ReLU`), adaptive pooling (`AdaptiveAvgPool1d`), and a linear classifier.
* **Dataset Split:** **80% Training / 20% Testing**
  * **Train Set (80%):** Used to iteratively learn feature representations and update model parameters.
  * **Test Set (20%):** Strictly isolated from optimization to evaluate unseen generalization.
* **Entropy Threshold:** `-150.0`

---

### 2/ Optimization Strategy

To optimize the network's trainable parameters (weights $W$ and biases $b$), we perform **Gradient Descent** on the loss landscape.

#### **The Loss Function (`nn.CrossEntropyLoss`)**
We measure classification error using **Cross-Entropy Loss**, which compares the predicted probability logits against the true target classes:

$$\mathcal{L}(y, \hat{y}) = -\sum_{c=1}^{C} y_c \log(\hat{y}_c)$$

* Unlike raw accuracy, Cross-Entropy Loss is a **smooth, continuous metric** that allows backpropagation to compute precise gradients.

**Backward Pass (`loss.backward()`):** Automatic differentiation computes the gradient ($\nabla \mathcal{L}$) indicating the uphill direction of error

$$w^{(t+1)} = w^{(t)} - \eta \cdot \nabla \mathcal{L}\left(w^{(t)}\right)$$
---
By repeating this cycle across the training set, the model progressively converges toward minimal classification error.


In [9]:
# Split: 80% training, 20% testing
size = len(list_entropy_vectors)
train_size = int(0.8 * size)
test_size = size - train_size

print(f"Total dataset size: {size}, Training size: {train_size}, Testing size: {test_size}")

# create the training and testing datasets
train_dataset = tr.DNAEntropyDataset(list_entropy_vectors[:train_size], threshold=-150.0)
test_dataset = tr.DNAEntropyDataset(list_entropy_vectors[train_size:], threshold=-150.0)

# create DataLoaders for training and testing
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

# Initialize the simple training model from training.py
model = tr.DNAEntropyCNN_Simple(num_classes=2)

# Calculate the number of parameters in the model
num_params = sum(p.numel() for p in model.parameters())
print(f"Number of parameters in the model: {num_params}")

# Calculates the loss function : comparing the predicted class probabilities with the true labels
#calculated during the training loop on the Training Set only
criterion = nn.CrossEntropyLoss()

# L'optimiseur Adam (adapte automatiquement les taux d'apprentissage)
#updates the weights of the model based on the computed gradients during backpropagation
#
optimizer = optim.Adam(model.parameters(), lr=0.001) 

# --------------------------------------------------------MODEL TRAINING-------------------------------------------------------

num_epochs = 10

print("---------------------------------------MODEL TRAINING-------------------------------------------------------")

for epoch in range(num_epochs):
    model.train()  
    total_loss = 0.0
    
    for batch_x, batch_y in train_loader:
        # initializing the gradients to zero before the backward pass
        optimizer.zero_grad()
        # predictions (logits) from the model for the current batch of input data
        logits = model(batch_x)
        
        # loss calculation
        loss = criterion(logits, batch_y)
        # PyTorch's automatic differentiation engine computes the gradients of the loss with respect to the model parameters
        loss.backward()
        # applying the updates to the model parameters based on the computed gradients
        optimizer.step()
        
        total_loss += loss.item() # keeping track of the cumulative loss for the epoch 
        
    loss_moyenne = total_loss / len(train_loader)
    print(f"Epoch [{epoch+1}/{num_epochs}] -> Average Loss  : {loss_moyenne:.4f}")

print("=============== Training completed ! ===============")


Total dataset size: 4554, Training size: 3643, Testing size: 911
Number of parameters in the model: 50
---------------------------------------MODEL TRAINING-------------------------------------------------------
Epoch [1/10] -> Average Loss  : 0.6089
Epoch [2/10] -> Average Loss  : 0.6040
Epoch [3/10] -> Average Loss  : 0.6046
Epoch [4/10] -> Average Loss  : 0.6040
Epoch [5/10] -> Average Loss  : 0.6047
Epoch [6/10] -> Average Loss  : 0.6041
Epoch [7/10] -> Average Loss  : 0.6043
Epoch [8/10] -> Average Loss  : 0.6038
Epoch [9/10] -> Average Loss  : 0.6045
Epoch [10/10] -> Average Loss  : 0.6036
=============== Training completed ! ===============


In [11]:
# ========== TESTING THE MODEL ON THE TEST SET
print("---------------------------------------MODEL TESTING-------------------------------------------------------")
model.eval()  #mode evaluation activated
correct_predictions = 0
total_samples = 0

# to economize memory and speed up computations, we disable gradient calculations during testing
with torch.no_grad():
    for batch_x, batch_y in test_loader:
        logits = model(batch_x)
        
        # Select the class with the highest score (argmax)
        predictions = torch.argmax(logits, dim=1)
        
        correct_predictions += (predictions == batch_y).sum().item()
        total_samples += batch_y.size(0)

accuracy = (correct_predictions / total_samples) * 100
print(f"\n Results on the test set (20%) :")
print(f"Accuracy : {accuracy:.2f}% ({correct_predictions}/{total_samples} correct sequences)")

---------------------------------------MODEL TESTING-------------------------------------------------------

 Results on the test set (20%) :
Accuracy : 71.13% (648/911 correct sequences)
